# Module 2 Coding Assignment [Optional]: Fine-Tuning Llama 2 (7 billion parameters) with VRAM Limitations and QLoRA:

In this notebook, we are going to fine-tune a Llama 2 model with 7 billion parameters on a small subset (https://huggingface.co/datasets/mlabonne/guanaco-llama2-1k) of the OpenAssistant Conversations (OASST1) dataset, a human-generated, human-annotated assistant-style conversation corpus. Note, this is mainly for you to understand LoRA and QLoRA and get experience working with the Hugging Face ecosystem. I encourage you to just follow along the code and observe

We will use Colab and therefore limited to a T4 GPU with 16 GB of VRAM. Given the VRAM limitations, traditional fine-tuning is not feasible, necessitating parameter-efficient fine-tuning (PEFT) techniques like LoRA or QLoRA. The chosen approach is QLoRA, which employs 4-bit precision to drastically reduce VRAM usage.

**Environment Setup:**
   - The task involves leveraging the Hugging Face ecosystem and several libraries: transformers, accelerate, peft, trl, and bitsandbytes.
   - You will need to add your hugging face token to colab secrets. Visit: https://huggingface.co/settings/tokens to set up your hugging face token, we are reading the LLama2 model only so you can set up a read only token. Then paste your token to your Colab Secret and name it "HF_TOKEN". See details here: https://x.com/GoogleColab/status/1719798406195867814

(Note: T4 GPU has 16 GB VRAM, 7 billion parameters of Llama 2 in 4-bit precision consume around 14 GB in FP16, and PEFT techniques like QLoRA are employed for efficient fine-tuning.)

## Step 1. Install Dependencies:

- Install required libraries (accelerate, peft, bitsandbytes, transformers, trl) using pip.

In [ ]:
!pip install -q -U transformers accelerate peft trl bitsandbytes

## Step 2. Import Libraries:

- Import necessary modules from installed libraries for model handling, tokenization, training, and PEFT.

In [ ]:
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    TextStreamer,
)
from peft import LoraConfig
from trl import SFTTrainer

## Step 3. Define Model, Dataset, and New Model Name:

- Specify the pre-trained model to use (model_name), the dataset for fine-tuning (dataset_name), and the desired name for your fine-tuned model (new_model).

In [ ]:
model_name = "NousResearch/Llama-2-7b-hf"
dataset_name = "mlabonne/guanaco-llama2-1k"
new_model = "llama-2-7b-miniguanaco"

In [ ]:
dataset = load_dataset(dataset_name, split="train")

README.md:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

data/train-00000-of-00001-9ad84bb9cf65a4(…): reconstructing file:   0%|          |  0.00B /  967kB            

data/train-00000-of-00001-9ad84bb9cf65a4(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

## Step 4. Configure QLoRA and BitsAndBytes:

- Set parameters for QLoRA (rank, alpha, dropout) to control the low-rank adaptation.
- Configure BitsAndBytes to load the model in 4-bit precision, reducing memory usage.
- Optionally, check if your GPU supports bfloat16 for potential training speedup.


In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
model.config.use_cache = False
model.config.pretraining_tp = 1

config.json:   0%|          | 0.00/583 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/26.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

tokenizer_config.json:   0%|          | 0.00/746 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

In [ ]:
from peft import prepare_model_for_kbit_training
import torch

model = prepare_model_for_kbit_training(model)

# Ubah semua parameter model dasar yang bertipe bfloat16 menjadi float16
for name, module in model.named_modules():
    if hasattr(module, "weight") and module.weight is not None:
        if module.weight.dtype == torch.bfloat16:
            module.to(torch.float16)

# Definisikan fungsi helper untuk memastikan tidak ada parameter latih (gradients) bfloat16 saat training dimulai
def cast_trainable_to_fp32(model):
    for name, param in model.named_parameters():
        if param.requires_grad:
            param.data = param.data.to(torch.float32)

In [ ]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

In [ ]:
training_arguments = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    optim="paged_adamw_32bit",
    save_steps=25,
    logging_steps=25,
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=True,
    bf16=False,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_steps=2,
    lr_scheduler_type="constant",
    report_to="tensorboard",
)

In [ ]:
from accelerate.state import AcceleratorState
if hasattr(AcceleratorState, '_shared_state') and AcceleratorState._shared_state:
    AcceleratorState._shared_state.clear()

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
    args=training_arguments,
)

/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py:148: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:331: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [ ]:
cast_trainable_to_fp32(trainer.model)

trainer.train()

/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
25,1.258322
50,1.205311
75,1.207203
100,1.263763
125,1.161307
150,1.156880
175,1.184618
200,1.203166
225,1.153027
250,1.131270


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

TrainOutput(global_step=250, training_loss=1.1924867095947265, metrics={'train_runtime': 2385.4524, 'train_samples_per_second': 0.419, 'train_steps_per_second': 0.105, 'total_flos': 2.2945460467777536e+16, 'train_loss': 1.1924867095947265, 'epoch': 1.0})

In [ ]:
trainer.model.save_pretrained(new_model)
tokenizer.save_pretrained(new_model)

('llama-2-7b-miniguanaco/tokenizer_config.json',
 'llama-2-7b-miniguanaco/tokenizer.json')

## Evaluate the finetuning result
Here's an interactive plot that can help you evaluate the finetuning results.

In [ ]:
def stream_inference(user_prompt: str):
    model.eval()

    system_prompt = (
        "Below is an instruction that describes a task. "
        "Write a response that appropriately completes the request.\n\n"
    )
    b_inst, e_inst = "### Instruction:\n", "### Response:\n"
    prompt = f"{system_prompt}{b_inst}{user_prompt.strip()}\n\n{e_inst}"

    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

    with torch.no_grad():
        _ = model.generate(
            **inputs,
            streamer=streamer,
            max_new_tokens=256,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.15,
        )

stream_inference("Tell me about operations research.")

Operations Research (OR) is a field of study that focuses on using mathematical and analytical techniques to solve complex problems in various fields, such as business, engineering, healthcare, and more. OR helps organizations make better decisions by providing insights into how different factors interact with each other to affect outcomes. By analyzing data from past experiences or current trends in the marketplace, OR can identify opportunities for improvement while minimizing risk associated with any given course of action. Through its use of statistical models combined with simulation tools like spreadsheets and computer programs designed specifically for this purpose; companies are able to optimize their processes efficiently resulting in increased profits over time due to reduced costs associated with inefficient production cycles among others things depending upon specific industry needs etcetera... 
